In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(("../data/raw/pubmed_papers.csv"))

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
embedding = model.encode(df['Abstract'].to_list(), show_progress_bar=True)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
df.shape

(100, 4)

In [6]:
embedding.shape

(100, 384)

In [7]:
query = 'lung cancer diagnosis biomarkers'

embedding_query = model.encode(query)

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity([embedding_query], embedding)

similarities.shape

(1, 100)

In [9]:
best_sim = similarities.argmax()
title = df.iloc[best_sim]['Title']
print(best_sim)
title

48


'Advances in early-stage lung cancer patients: from preanalytics to molecular analysis.'

In [10]:
top_5_best_indx = np.argsort(similarities[0])[-5:][::-1]
top_5_best_indx

array([48, 92, 16, 61, 81])

In [ ]:
for rank, idx in enumerate(top_5_best_indx, start=1):
    title = df.iloc[idx]['Title']
    print(f"{rank}.{title}")
    
    sim = similarities[0][idx]
    print(f"Similarity: {sim:.4f}\n")
    rank +=1
    

1.Advances in early-stage lung cancer patients: from preanalytics to molecular analysis.
Similarity: 0.6935

2.Identification of High-Performing Blood Metabolite Biomarkers of Lung Cancer in a Chinese Population.
Similarity: 0.6737

3.Combining Commercial Cancer Protein Biomarkers and Benign Fungal Antibodies Improves Diagnostic Accuracy in Pulmonary Nodules.
Similarity: 0.6388

4.Multidimensional proteomics and explainable AI feature selection identify cross-platform lung cancer molecular signature in blood plasma.
Similarity: 0.6368

5.Association between tissue origin classification and the clinical features and molecular phenotypes of nonsmall cell lung cancer.
Similarity: 0.6348



In [17]:
def search_paper(query, top_k):
    embedding_query = model.encode(query)
    similarities = cosine_similarity([embedding_query], embedding)
    top_k_indx = np.argsort(similarities[0])[-top_k:][::-1]
    results= []
    
    for idx in top_k_indx:
        pmid = int(df.iloc[idx]['PMID'])
        title = df.iloc[idx]['Title']
        abstract = df.iloc[idx]['Abstract']
        sim = float(similarities[0][idx])

        results.append({
            'PMID':pmid,
            'Title':title,
            'Abstract':abstract,
            'Similarities':sim
            })
          
    return(results)

result = search_paper("lung cancer diagnosis biomarkers", 3)
print(result)

    


[{'PMID': 42252070, 'Title': 'Advances in early-stage lung cancer patients: from preanalytics to molecular analysis.', 'Abstract': 'Lung Cancer (LC) remains one of the leading causes of cancer death worldwide. Molecular profiling of mandatory testing biomarkers significantly meliorates clinical outcome of advanced stage LC patients while intercepting early-stage lesions represents an unmet need. Recent advances in treatments for early-stage LC patients lay the basis for accelerating molecular analysis of actionable drivers before metastatic stages. In this scenario, liquid biopsy emerged as an innovative tool to guide clinical decision-making processes of early-stage LC patients. In addition, digital pathology is pioneering an integrative path to optimize morpho-molecular analysis. Undoubtedly, the lack of standardized preanalytical procedures significantly reduces the clinical availability of these approaches. This review aims to explore critical gaps affecting the clinical implementa